<a href="https://colab.research.google.com/github/pkocmann/messyverse/blob/main/notebooks/04_datei-sortieren.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab öffnen"/></a>

# Den Datei-Zoo ordnen -- nach Datum

Im `magazin/` liegen digitalisierte Dokumente mit gewachsenen Namen (`Scan_2025_03_final_FINAL_v2.pdf`, `dokument (3).pdf`, `IMG_0421.pdf` ...). Dein Auftrag: zu jeder Datei das Datum bestimmen (JJJJ-MM-TT). Manche Daten stehen im Namen, manche nur im Inhalt -- und einige Dateien lassen sich **nicht** datieren. Die kommen in den Sammelordner 'undatiert'. Eine Datei ist sogar leer (0 Byte).

In [ ]:
# SETUP (Black Box -- einmal ausführen). Holt deine Arbeitskopie des Übungsuniversums.
import os, shutil, subprocess
ZIEL = "/content/messyverse"
os.chdir("/content")                 # erst aus ZIEL heraus -- sonst löscht der Re-Lauf das eigene Arbeitsverzeichnis
if os.path.exists(ZIEL):
    shutil.rmtree(ZIEL)              # idempotent: alte Kopie weg, dann frisch klonen
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/pkocmann/messyverse.git", ZIEL], check=True)
os.chdir(ZIEL)
anzahl = sum(len(d) for r, _, d in os.walk(ZIEL) if ".git" not in r)
print(f"Arbeitskopie: {anzahl} Dateien unter {ZIEL}")

In [ ]:
!pip install -q pypdf

## Schritt 1 -- die Lage ansehen

Hinweis: das verlässliche Tagesdatum steht in der Zeile `Datum:` im PDF-Inhalt; die Dateinamen tragen teils nur Monat/Jahr.

In [ ]:
import os
for n in sorted(os.listdir("magazin")):
    print(os.path.getsize(os.path.join("magazin", n)), "Byte  ", n)

## Schritt 2 -- die KI prompten

> *Bestimme für jede Datei in `magazin/` das Datum als JJJJ-MM-TT. Nutze bevorzugt den PDF-Inhalt (pypdf, Zeile 'Datum: JJJJ-MM-TT'); der Dateiname hilft zusätzlich. Dateien ohne erkennbares Datum und leere (0-Byte-)Dateien bekommen den Wert 'undatiert'. Baue ein dict `ergebnis`: Dateiname -> Datum bzw. 'undatiert'.*

In [ ]:
# Code deines KI-Assistenten hier einfügen.
# Ziel: dict `ergebnis` -> 'dateiname.pdf': 'JJJJ-MM-TT' oder 'undatiert'


## Schritt 3 -- gegen das Universum prüfen (Black Box)

Das Soll-Datum steht ohnehin im öffentlichen Datei-Inhalt; die Zelle nennt nur, **wo** dein Mapping abweicht.

In [ ]:
import json
try:
    soll = json.load(open("loesungen/datei_sortieren.golden.json"))["mapping"]
except FileNotFoundError:
    print("Bitte zuerst die SETUP-Zelle ausführen (oben).")
else:
    try:
        ergebnis
    except NameError:
        print("Noch kein `ergebnis`.")
    else:
        if not isinstance(ergebnis, dict):
            print(f"Dein `ergebnis` ist gerade ein {type(ergebnis).__name__}, erwartet wird ein dict Dateiname -> Datum.")
        else:
            fehlen = [n for n in soll if n not in ergebnis]
            ab = [n for n in soll if n in ergebnis and str(ergebnis[n]) != soll[n]]
            extra = [n for n in ergebnis if n not in soll]
            if not (fehlen or ab or extra):
                print(f"OK -- alle {len(soll)} Dateien korrekt datiert (inkl. undatiert).")
            else:
                if fehlen: print("Diese Dateien fehlen im Mapping:", fehlen)
                if ab: print("Bei diesen Dateien weicht das Datum ab:", ab)
                if extra: print("Diese Einträge gehören nicht ins Mapping (unbekannte Datei):", extra)

## Wenn es klemmt -- die Fehlerschleife

Lief der Code auf einen Fehler, oder meldet die Prüfzelle Abweichungen? Dann **nicht selbst reparieren**. Kopiere die Fehlermeldung oder die Abweichungs-Liste, gib sie deinem Assistenten zurück und bitte um eine korrigierte Fassung. Die häufigsten Ursachen sind übersehene Sonderfälle -- leere Einträge, fehlende Felder, ungewohnte Schreibweisen.

## Weiterdenken

Hat die KI die leere Datei und `IMG_0421.pdf` korrekt als 'undatiert' behandelt -- oder ist sie an der 0-Byte-Datei abgestürzt? Genau solche Sonderfälle übersieht generierter Code häufig. Als Kür: lass die Dateien tatsächlich in Unterordner je Monat verschieben, 'undatiert' separat.